====================================================================
COSC 2060: Modern Data Engineering & AI Infrastructure
Unit 5 Synchronous Coding Clinic ("I Do, We Do, You Do")
Topic: Data as a Service (FastAPI & Pydantic v2 Contracts)
====================================================================
Run each cell below one at a time (`Shift + Enter`) to inspect outputs step by step.
====================================================================


### PART 0: BASICS WARM-UP — MINIMAL PYDANTIC MODEL & VALIDATION CHECK


In [ ]:
print("\n>>> Executing PART 0: BASICS WARM-UP (Minimal Pydantic BaseModel)...")
from pydantic import BaseModel, Field, ValidationError, field_validator

class BasicTruckCheck(BaseModel):
    truck_num: int = Field(gt=0)
    city: str = Field(min_length=2)

warm_truck = BasicTruckCheck(truck_num=101, city="Cheyenne")
print("Part 0 Warm-Up — Validated dict:", warm_truck.model_dump())
try:
    BasicTruckCheck(truck_num=-5, city="Cheyenne")
except ValidationError as err:
    print("Part 0 Warm-Up — Caught negative truck_num:", err.errors()[0]["msg"])


### PART 1: 'I DO' — INSTRUCTOR DEMONSTRATION


In [ ]:
print("\n>>> Executing PART 1: 'I DO' — INSTRUCTOR DEMONSTRATION...")

class DeliveryWindow(BaseModel):
    shipment_id: str = Field(..., pattern=r"^SHP-\d{5}$")
    transit_hours: float = Field(..., gt=0.0, le=72.0)
    carrier_rating: float = Field(..., ge=1.0, le=5.0)

    @field_validator("transit_hours")
    @classmethod
    def check_realistic_transit(cls, v: float) -> float:
        if v < 0.5:
            raise ValueError("Transit hours cannot be under 30 minutes for interstate freight.")
        return v

# Valid instance:
valid_win = DeliveryWindow(shipment_id="SHP-10023", transit_hours=14.5, carrier_rating=4.8)
print("Valid model created successfully:", valid_win.model_dump())


### PART 2: 'WE DO' — GUIDED CLASS PRACTICE


In [ ]:
print("\n>>> Executing PART 2: 'WE DO' — GUIDED CLASS PRACTICE...")
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

demo_app = FastAPI()

@demo_app.get("/v1/ping")
def ping():
    return {"status": "healthy", "service": "telemetry-gateway"}

client = TestClient(demo_app)
response = client.get("/v1/ping")
assert response.status_code == 200
assert response.json()["status"] == "healthy"
print("Ping endpoint passed verification!")
